In [45]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import pandas as pd

In [46]:
# Load dataset
file_path = ('/content/CASP.csv')
data = pd.read_csv(file_path)


In [47]:
# Prepare data
X = data.iloc[:, 1:].values  # Features (F1 to F9)
y = data.iloc[:, 0].values  # Target (RMSD)

# Ensure y is numeric
y = np.array(y, dtype=np.float32)  # Directly convert to numpy array with dtype float32

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)


In [48]:
# Prepare DataLoader
def create_dataloader(X, y, batch_size):
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Define Vanilla MLP Model
class VanillaMLP(nn.Module):
    def __init__(self, input_size, hidden_layers, hidden_neurons, activation):
        super(VanillaMLP, self).__init__()
        layers = []
        in_features = input_size
        for _ in range(hidden_layers):
            layers.append(nn.Linear(in_features, hidden_neurons))
            layers.append(activation)
            in_features = hidden_neurons
        layers.append(nn.Linear(in_features, 1))  # Output layer
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [49]:
# Experiment functions

def train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        predictions = model(X_test_tensor)
        mse = mean_squared_error(y_test_tensor.numpy(), predictions.numpy())
    return mse


In [50]:
def experiment_hidden_layers(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, neurons, activation, epochs, lr, batch_size):
    results = []
    for hidden_layers in [1, 2, 3]:
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Hidden Layers': hidden_layers, 'MSE': mse})
    print("Hidden Layers Experiment Results:")
    for result in results:
        print(f"Hidden Layers: {result['Hidden Layers']}\nMSE: {result['MSE']}\n")
    return results

In [51]:
def experiment_neurons(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, hidden_layers, activation, epochs, lr, batch_size):
    results = []
    for neurons in [4, 8, 16, 32, 64]:
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Neurons': neurons, 'MSE': mse})
    print("Neurons Experiment Results:")
    for result in results:
        print(f"Neurons: {result['Neurons']}\nMSE: {result['MSE']}\n")
    return results

In [52]:
def experiment_activation(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, hidden_layers, neurons, epochs, lr, batch_size):
    results = []
    for activation_name, activation in {'linear': nn.Identity(), 'sigmoid': nn.Sigmoid(), 'relu': nn.ReLU(), 'softmax': nn.Softmax(dim=1), 'tanh': nn.Tanh()}.items():
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Activation': activation_name, 'MSE': mse})
    print("Activation Experiment Results:")
    for result in results:
        print(f"Activation: {result['Activation']}\nMSE: {result['MSE']}\n")
    return results

In [41]:
def experiment_epochs(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, hidden_layers, neurons, activation, lr, batch_size):
    results = []
    for epochs in [1, 10, 25, 50, 100, 250]:
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Epochs': epochs, 'MSE': mse})
    print("Epochs Experiment Results:")
    for result in results:
        print(f"Epochs: {result['Epochs']}\nMSE: {result['MSE']}\n")
    return results

In [54]:
def experiment_learning_rate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, hidden_layers, neurons, activation, epochs, batch_size):
    results = []
    for lr in [10, 1, 0.1, 0.01, 0.001, 0.0001]:
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Learning Rate': lr, 'MSE': mse})
    print("Learning Rate Experiment Results:")
    for result in results:
        print(f"Learning Rate: {result['Learning Rate']}\nMSE: {result['MSE']}\n")
    return results

In [55]:
def experiment_batch_size(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, hidden_layers, neurons, activation, epochs, lr):
    results = []
    for batch_size in [16, 32, 64, 128, 256, 512]:
        train_loader = create_dataloader(X_train_tensor, y_train_tensor, batch_size)
        model = VanillaMLP(X_train_tensor.shape[1], hidden_layers, neurons, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        mse = train_and_evaluate(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, model, criterion, optimizer, epochs)
        results.append({'Batch Size': batch_size, 'MSE': mse})
    print("Batch Size Experiment Results:")
    for result in results:
        print(f"Batch Size: {result['Batch Size']}\nMSE: {result['MSE']}\n")
    return results

# Example: Running experiments for hidden layers
hidden_layers_results = experiment_hidden_layers(X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, neurons=16, activation=nn.ReLU(), epochs=50, lr=0.01, batch_size=64)


Hidden Layers Experiment Results:
Hidden Layers: 1
MSE: 43.53614807128906

Hidden Layers: 2
MSE: 31.317291259765625

Hidden Layers: 3
MSE: 31.039695739746094

